[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlos-argueta/factor-graphs-from-scratch/blob/main/modules/00-ekf-to-least-squares/solutions/exercises_solutions.ipynb)

# Module 0 — Exercise solutions

In [1]:
import sys, pathlib, subprocess
import numpy as np

# --- Colab bootstrap -------------------------------------------------------
# On Google Colab the repository isn't present, so clone it once to make the
# course packages (`gaussian_filters`, `fgslam`) importable.
# EDIT REPO_URL to point at your fork.
REPO_URL = "https://github.com/carlos-argueta/factor-graphs-from-scratch.git"
if "google.colab" in sys.modules:
    repo = pathlib.Path("/content/factor-graphs-from-scratch")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=True)
    if str(repo / "src") not in sys.path:
        sys.path.insert(0, str(repo / "src"))

# --- Local: put the repo's src/ on the path (covers gaussian_filters + fgslam)
def _add_src_to_path():
    here = pathlib.Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "src" / "gaussian_filters").exists():
            if str(base / "src") not in sys.path:
                sys.path.insert(0, str(base / "src"))
            return
_add_src_to_path()

np.set_printoptions(precision=5, suppress=True)

### E0.1 — sequential & order-independent updates

In [2]:
L = np.array([5.0, 3.0]); mu0 = np.array([1.0, 1.0]); Sigma0 = np.diag([0.5,0.8])**2
sig_r, sig_b = 0.10, 0.05
def h(mu):
    dx,dy = L-mu; return np.array([np.hypot(dx,dy), np.arctan2(dy,dx)])
def H(mu):
    dx,dy = L-mu; r2=dx*dx+dy*dy; r=np.sqrt(r2)
    return np.array([[-dx/r,-dy/r],[dy/r2,-dx/r2]])
z = h(np.array([1.3,0.7])) + np.array([0.04,-0.02])

def fuse(mu, Sig, zi, hi, Hi, Qi):
    Hm = Hi(mu); S = Hm@Sig@Hm.T + Qi; K = Sig@Hm.T@np.linalg.inv(S)
    mu = mu + K@(zi - hi(mu)); Sig = (np.eye(len(mu))-K@Hm)@Sig
    return mu, Sig

# combined
muC,SigC = fuse(mu0,Sigma0,z,h,H,np.diag([sig_r,sig_b])**2)
# range then bearing
hr=lambda m:h(m)[:1]; Hr=lambda m:H(m)[:1]
hb=lambda m:h(m)[1:]; Hb=lambda m:H(m)[1:]
m,S = fuse(mu0,Sigma0,z[:1],hr,Hr,np.array([[sig_r**2]]))
m,S = fuse(m,S,z[1:],hb,Hb,np.array([[sig_b**2]]))
# bearing then range
m2,S2 = fuse(mu0,Sigma0,z[1:],hb,Hb,np.array([[sig_b**2]]))
m2,S2 = fuse(m2,S2,z[:1],hr,Hr,np.array([[sig_r**2]]))
print("combined :", muC)
print("r then b :", m, " allclose:", np.allclose(muC,m,atol=1e-6))
print("b then r :", m2, " allclose:", np.allclose(muC,m2,atol=1e-6))

combined : [1.19298 0.77108]
r then b : [1.19124 0.77452]  allclose: False
b then r : [1.20026 0.77703]  allclose: False


Sequential = combined because each update *adds* its information block, and addition commutes.
The tiny order-dependence comes only from re-linearising $H$ at a slightly moved mean; the move
is sub-centimetre here so it's negligible. (For strongly nonlinear $h$ or large innovations the
order can matter at the linearisation-error level — a reason batch re-linearisation, M3+, wins.)

### E0.2 — third factor (GPS)

In [3]:
R = np.diag([0.2,0.2])**2; z_gps = np.array([1.28,0.66])
# sequential: continue from the combined range-bearing posterior
mg,Sg = fuse(muC,SigC,z_gps,lambda m:m,lambda m:np.eye(2),R)
# batch 3-factor: information add
Lam = np.linalg.inv(Sigma0) + H(mu0).T@np.linalg.inv(np.diag([sig_r,sig_b])**2)@H(mu0) \
      + np.eye(2).T@np.linalg.inv(R)@np.eye(2)
# linearised one-shot from mu0 (GN step with all three residuals)
rhs = H(mu0).T@np.linalg.inv(np.diag([sig_r,sig_b])**2)@(z-h(mu0)) \
      + np.linalg.inv(R)@(z_gps-mu0)
mu_batch = mu0 + np.linalg.solve(Lam, rhs)
print("sequential:", mg)
print("batch 1-GN:", mu_batch, " (small diff = relinearisation of range-bearing)")
print("covariance match:", np.allclose(Sg, np.linalg.inv(Lam), atol=1e-9))

sequential: [1.23019 0.70766]
batch 1-GN: [1.23019 0.70766]  (small diff = relinearisation of range-bearing)
covariance match: True


The prior for the GPS update is the range–bearing posterior. Covariances match exactly (linear GPS); means differ only by the range–bearing relinearisation point.

### E0.3 — information add / subtract

In [4]:
Hm = H(mu0); Qi = np.diag([sig_r,sig_b])**2
Lam_post = np.linalg.inv(Sigma0) + Hm.T@np.linalg.inv(Qi)@Hm
Lam_back = Lam_post - Hm.T@np.linalg.inv(Qi)@Hm
print("recovered prior info == Sigma0^-1 :", np.allclose(Lam_back, np.linalg.inv(Sigma0)))

recovered prior info == Sigma0^-1 : True


Subtracting $H^\top Q^{-1}H$ from the information matrix exactly recovers the prior information —
information is linearly additive, so "un-adding" is trivial. In **covariance** form there is no
clean subtraction (you'd need the Sherman–Morrison/Woodbury identity and it can leave an invalid,
non-PSD matrix). This asymmetry is exactly why marginalisation and incremental smoothing are done
in the **information** form — the heart of M2 and M5.